# Lab 07: Diffusion Visualization

**Module 07** | Read `notes/07-diffusion-theory.pdf` before running this notebook.

Visualize the forward noising process and a linear beta schedule on MNIST digits, the foundation for DDPM training.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Forward diffusion on MNIST

Denoising diffusion models gradually corrupt data by adding Gaussian noise over `T` timesteps. The **forward process** is Markovian: at each step a small amount of noise is blended in according to a variance schedule `β_t`. After enough steps the image approaches pure noise.

Understanding the forward process is prerequisite to DDPM training in Lab 08: the network learns to reverse these corruption steps.


We use a **linear beta schedule**, `β_t` increases evenly from a small start value to a larger end value. From `β_t` we derive `α_t = 1 − β_t` and the cumulative product `ᾱ_t = ∏ α_i`, which lets us jump directly from clean image `x_0` to any noisy `x_t` in closed form:

`x_t = √(ᾱ_t) · x_0 + √(1 − ᾱ_t) · ε` with `ε ~ N(0, I)`.

The plot below shows how `β_t` and `ᾱ_t` evolve across timesteps.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
T = 500

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(betas.numpy())
axes[0].set_title("Linear β schedule")
axes[0].set_xlabel("Timestep t")
axes[0].set_ylabel("β_t")
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_bars.numpy())
axes[1].set_title("Cumulative ᾱ_t = ∏(1 − β)")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("ᾱ_t")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"T = {T} | β_1 = {betas[0]:.6f} | β_T = {betas[-1]:.4f}")
print(f"ᾱ_0 = {alpha_bars[0]:.4f} | ᾱ_{T-1} = {alpha_bars[-1]:.6f}")


We load one MNIST digit and apply the closed-form forward diffusion formula at timesteps `[0, 50, 100, 200, 500]`. Early steps look almost unchanged; by step 500 the structure dissolves into noise. This visual confirms why the denoising network must handle a wide range of signal-to-noise ratios.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
x0, label = train_set[0]
x0 = x0.unsqueeze(0).to(device)
print(f"Selected digit label: {label}")

timesteps = [0, 50, 100, 200, 500]
noisy_images = []

torch.manual_seed(42)
noise = torch.randn_like(x0)

for t in timesteps:
    if t == 0:
        xt = x0
    else:
        ab = alpha_bars[t - 1].to(device)
        xt = torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * noise
    noisy_images.append(xt.cpu())

fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 3))
for ax, t, img in zip(axes, timesteps, noisy_images):
    disp = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(disp, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t = {t}")
    ax.axis("off")
plt.suptitle("Forward diffusion of one MNIST digit")
plt.tight_layout()
plt.show()
